In [ ]:
# FL CREDIT CARD FRAUD DETECTION
import sys
!{sys.executable} -m pip install imbalanced-learn

import os
import random
import numpy as np
import torch

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(torch.cuda.device_count() > 0)

set_seed(42)

############################################################
# IMPORTS
############################################################
from imblearn.over_sampling import SMOTE
import torch.nn as nn
import torch.optim as optim
import pandas as pd
import copy
import pickle
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_score, recall_score, roc_auc_score, fbeta_score
from sklearn.preprocessing import StandardScaler
from torch.utils.data import DataLoader, TensorDataset

# pca
from sklearn.decomposition import PCA

############################################################
# DATA
############################################################

def load_dataset():

    url = "https://storage.googleapis.com/download.tensorflow.org/data/creditcard.csv"
    file_path = "creditcard.csv"

    # Download automatically if not present
    if not os.path.exists(file_path):
        print("⬇️ Downloading dataset automatically...")
        import urllib.request
        urllib.request.urlretrieve(url, file_path)
        print("✅ Download complete")
    else:
        print("✅ Dataset already exists")

    # Load dataset
    df = pd.read_csv(file_path)

    X = df.drop(["Class"], axis=1).values
    y = df["Class"].values

    # Train-test split
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, stratify=y, random_state=42
    )

    # Scaling
    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_test = scaler.transform(X_test)

    # # SMOTE balancing
    # sm = SMOTE(random_state=42)
    # X_train, y_train = sm.fit_resample(X_train, y_train)

    # print("📊 Data ready:", X_train.shape)

    return X_train, X_test, y_train, y_test


def smote():
    X_train, X_test, y_train, y_test = load_dataset()
    sm = SMOTE(random_state=42)
    X_train, y_train = sm.fit_resample(X_train, y_train)
    return X_train, X_test, y_train, y_test

############################################################
# DIRICHLET
############################################################

def dirichlet_partition(X, y, num_clients=20, alpha=0.5):

    classes = np.unique(y)
    client_data = [[] for _ in range(num_clients)]

    for c in classes:
        idx_c = np.where(y == c)[0]
        np.random.shuffle(idx_c)

        proportions = np.random.dirichlet(alpha * np.ones(num_clients))
        proportions = proportions / proportions.sum()

        proportions = (proportions * len(idx_c)).astype(int)
        diff = len(idx_c) - proportions.sum()
        proportions[0] += diff

        start = 0
        for i in range(num_clients):
            end = start + proportions[i]
            client_data[i].extend(idx_c[start:end])
            start = end

    return [(X[idx], y[idx]) for idx in client_data]


def get_clients(X, y):
    try:
        with open("clients.pkl", "rb") as f:
            clients = pickle.load(f)
        print("Loaded saved partition ✅")
    except:
        clients = dirichlet_partition(X, y)
        with open("clients.pkl", "wb") as f:
            pickle.dump(clients, f)
        print("Saved new partition ✅")
    return clients

############################################################
# MODEL
############################################################

class FraudModel(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 1)
        )

    def forward(self, x):
        return self.net(x)

############################################################
# LOSS
############################################################

class FocalLoss(nn.Module):
    def __init__(self, alpha=0.25, gamma=2):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma

    def forward(self, logits, targets):
        bce = nn.functional.binary_cross_entropy_with_logits(logits, targets, reduction='none')
        prob = torch.sigmoid(logits)
        pt = prob * targets + (1 - prob) * (1 - targets)
        return (self.alpha * (1 - pt) ** self.gamma * bce).mean()

############################################################
# VAE
############################################################

class VAE(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.encoder = nn.Sequential(nn.Linear(input_dim, 128), nn.ReLU(), nn.Linear(128, 64))
        self.mu = nn.Linear(64, 16)
        self.logvar = nn.Linear(64, 16)
        self.decoder = nn.Sequential(nn.Linear(16, 64), nn.ReLU(), nn.Linear(64, input_dim))

    def forward(self, x):
        h = self.encoder(x)
        mu, logvar = self.mu(h), self.logvar(h)
        std = torch.exp(0.5 * logvar)
        z = mu + torch.randn_like(std) * std
        return self.decoder(z), mu, logvar

def vae_loss(x, recon, mu, logvar):
    return ((x - recon) ** 2).mean() + 0.01 * (-0.5 * torch.mean(1 + logvar - mu.pow(2) - logvar.exp()))

############################################################
# UTIL
############################################################

def flatten_weights(weights):
    return torch.cat([v.flatten() for v in weights.values()])

############################################################
# TRAIN
############################################################

def train_client(model, loader, global_weights):
    model.load_state_dict(global_weights)
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    criterion = FocalLoss()

    for x, y in loader:
        pred = model(x).squeeze(-1) # Changed .squeeze() to .squeeze(-1)
        loss = criterion(pred, y.float())
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    return model.state_dict()

############################################################
# ATTACKS (IMPROVED)
############################################################
def malicious(model, loader, global_w):
    model.load_state_dict(global_w)
    opt = optim.Adam(model.parameters(), lr=1e-3)

    for x,y in loader:
        y = 1 - y
        loss = nn.BCEWithLogitsLoss()(model(x).squeeze(-1), y.float()) # Changed .squeeze() to .squeeze(-1)
        opt.zero_grad()
        loss.backward()
        opt.step()

    w = model.state_dict()
    # return {k: global_w[k] + 2*(w[k]-global_w[k]) for k in w}
    # return {k: global_w[k] + 0.5*(w[k]-global_w[k]) for k in w}
    return {k: global_w[k] + 0.3*(w[k]-global_w[k]) for k in w}


def stealth(model, loader, global_w):
    model.load_state_dict(global_w)
    opt = optim.Adam(model.parameters(), lr=1e-3)

    for x,y in loader:
        mask = torch.rand_like(y.float()) < 0.1
        y[mask] = 1 - y[mask]
        loss = nn.BCEWithLogitsLoss()(model(x).squeeze(-1), y.float()) # Changed .squeeze() to .squeeze(-1)
        opt.zero_grad()
        loss.backward()
        opt.step()

    w = model.state_dict()
    # return {k: w[k] + 0.2*torch.sign(w[k]) for k in w}
    return {k: global_w[k] + 0.2*(w[k] - global_w[k]) for k in w}



###############################
#
def directional_drift_attack(model, loader, global_weights, epsilon=0.02):

    model.load_state_dict(global_weights)
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    criterion = FocalLoss()

    # Normal training
    for x, y in loader:
        pred = model(x).squeeze(-1) # Changed .squeeze() to .squeeze(-1)
        loss = criterion(pred, y.float())
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    weights = model.state_dict()

    # Compute update
    update = {k: weights[k] - global_weights[k] for k in weights}

    # Inject directional bias
    poisoned = {}
    for k in update:
        direction = torch.sign(update[k])
        poisoned[k] = update[k] + epsilon * direction

    # Return poisoned weights
    return {k: global_weights[k] + poisoned[k] for k in poisoned}


############################################################
# PCA VISUALIZATION
############################################################
def plot_pca(vectors, labels, title):
    vectors = np.array(vectors)
    labels = np.array(labels)

    pca = PCA(n_components=2)
    reduced = pca.fit_transform(vectors)

    plt.figure()
    plt.scatter(reduced[labels==0][:,0], reduced[labels==0][:,1], label="Normal")
    plt.scatter(reduced[labels==1][:,0], reduced[labels==1][:,1], label="Attack", marker='x')
    plt.title(title)
    plt.legend()
    plt.grid()
    plt.show()

############################################################
# AGGREGATION
############################################################

def trimmed_mean(updates):


    # stacked = torch.stack(updates)
    # sorted_updates, _ = torch.sort(stacked, dim=0)
    # trim = int(0.2 * len(updates))
    # return torch.mean(sorted_updates[trim:-trim], dim=0)

  #
    n = len(updates)
  #
    stacked = torch.stack(updates)
    sorted_updates, _ = torch.sort(stacked, dim=0)
    trim = min(int(0.2 * n), (n - 1) // 2)

    return torch.mean(sorted_updates[trim:-trim], dim=0)


def fedavg(updates):
    return torch.mean(torch.stack(updates), dim=0)


############################################################
# EVALUATION
############################################################

def find_best_threshold(y_true, prob):
    best_t, best_score = 0.5, 0
    for t in np.linspace(0.5, 0.99, 50):
        pred = (prob > t).astype(int)
        score = fbeta_score(y_true, pred, beta=0.5, zero_division=0)
        if score > best_score:
            best_score, best_t = score, t
    return best_t

def evaluate(model, X_test, y_test):
    model.eval()
    with torch.no_grad():
        prob = torch.sigmoid(model(torch.tensor(X_test).float()).squeeze()).numpy()

    prob = np.clip(prob, 0, 0.99)
    t = find_best_threshold(y_test, prob)
    pred = (prob > t).astype(int)

    return (
        precision_score(y_test, pred, zero_division=0),
        recall_score(y_test, pred, zero_division=0),
        roc_auc_score(y_test, prob),
        t
    )

############################################################
# EXPERIMENT
############################################################

def run_experiment(use_trimmed=True, use_vae=False, use_rollback=False, use_multi_krum=False ):

  #
    precision_hist, recall_hist, auc_hist, asr_hist = [], [], [], []
    detection_rate_hist = []

    avg_scores_hist = []
    threshold_hist = []
  #
    X_train, X_test, y_train, y_test = load_dataset()
    clients = dirichlet_partition(X_train, y_train)

    global_model = FraudModel(X_train.shape[1])
    server_weights = global_model.state_dict()

    vae = VAE(len(flatten_weights(server_weights)))
    vae_opt = optim.Adam(vae.parameters(), lr=0.001)
    vae_threshold = None


    for round in range(40):

        updates, vecs = [], []
        attack_flags = []

        total_attacks = 0
        detected_attacks = 0
        bypassed_attacks = 0

        prev_w = copy.deepcopy(server_weights)

        for cid,(Xc,yc) in enumerate(clients):

            # 🔥 per-client SMOTE
            unique_classes, counts = np.unique(yc, return_counts=True)
            if len(unique_classes) > 1:
                minority_count = min(counts)
                if minority_count > 1:
                    k_neighbors_val = min(minority_count - 1, 5)
                    sm = SMOTE(random_state=42, k_neighbors=k_neighbors_val)
                    Xc,yc = sm.fit_resample(Xc,yc)
                # else: if minority_count is 1, SMOTE cannot be applied for this class.
            # else: SMOTE is skipped for clients with only one class.

            loader = DataLoader(TensorDataset(
                torch.tensor(Xc).float(),
                torch.tensor(yc)
            ), batch_size=128)

            model = FraudModel(X_train.shape[1])

            is_attack = False


            #  MULTI-CLIENT  ATTACK

            if cid==0 and round>23:
                w = malicious(model, loader, server_weights)
                is_attack=True
                print("🚨 malicious attack triggered")
            elif cid==3 and round>25:
                w = stealth(model, loader, server_weights)
                is_attack=True
                print("🚨 stealth attack triggered")
            else:
                w = train_client(model, loader, server_weights)

            if is_attack:
                total_attacks += 1

            update = {k: w[k]-server_weights[k] for k in server_weights}
            vec = flatten_weights(update)

            updates.append(update)
            vecs.append(vec)
            attack_flags.append(is_attack)

            # VAE TRAIN ONLY CLEAN
            if use_vae:
                # if round < 10 and not is_attack:
                # if round < 15 and not is_attack:
                if round < 18 and not is_attack:
                    recon, mu, logvar = vae(vec)
                    loss = vae_loss(vec, recon, mu, logvar)
                    vae_opt.zero_grad()
                    loss.backward()
                    vae_opt.step()


        ##################################################
        # VAE DETECTION (improved with latent space)
        ##################################################
        trusted = updates

        if use_vae:

            vae.eval()
            scores = []
            norms = torch.tensor([torch.norm(v) for v in vecs])
            median = torch.median(norms)

            # STEP 1: raw values
            raw_recon, raw_latent = [], []
            for vec in vecs:
                recon, mu, _ = vae(vec)
                raw_recon.append(torch.mean((vec - recon)**2).item())
                raw_latent.append(torch.norm(mu).item())

            # STEP 2: normalized scores
            scores = []
            mean_recon = np.mean(raw_recon)
            mean_latent = np.mean(raw_latent)

            for i in range(len(vecs)):

              recon_norm = (raw_recon[i] - np.mean(raw_recon)) / (np.std(raw_recon) + 1e-8)
              latent_norm = (raw_latent[i] - np.mean(raw_latent)) / (np.std(raw_latent) + 1e-8)

              # score = recon_norm + 0.5 * latent_norm
              # score = recon_norm + 0.6 * latent_norm
              # score = recon_norm + 0.7 * latent_norm
              score = recon_norm + 0.8 * latent_norm

              scores.append(score)

##########
            #  Track average anomaly score
            avg_scores_hist.append(np.mean(scores))

##########

            # if round == 15:
            if round == 20:
               
                # vae_threshold = np.mean(scores) + 0.7 * np.std(scores)
                # vae_threshold = np.mean(scores) + 0.65 * np.std(scores)
                vae_threshold = np.mean(scores) + 0.7 * np.std(scores)

            if round > 25:
                # vae_threshold = 0.95 * vae_threshold + 0.05 * (np.mean(scores) + 0.8*np.std(scores))
                vae_threshold = 0.95 * vae_threshold + 0.05 * (np.mean(scores) + 0.7*np.std(scores))          #


            # threshold = np.percentile(scores, 90)
            threshold = vae_threshold if vae_threshold is not None else np.percentile(scores, 90)

            threshold_hist.append(threshold)
 

            # ===============================
            # STORE DATA ONLY (NO PLOT)
            # ===============================
            if round == 24:
                saved_normal_scores = [scores[i] for i in range(len(scores)) if not attack_flags[i]]
                saved_attack_scores = [scores[i] for i in range(len(scores)) if attack_flags[i]]

                saved_threshold = threshold  

            filtered_updates = []


            for i in range(len(vecs)):

                keep = True

                score_flag = scores[i] > threshold
                # norm_flag = norms[i] > 1.4 * median
                norm_flag = norms[i] > 1.5 * median

             
                if round > 18:
                  if score_flag or norm_flag:
                      keep = False


                if keep:
                    filtered_updates.append(updates[i])
                    if attack_flags[i]:
                        bypassed_attacks += 1
                else:
                    if attack_flags[i]:
                        detected_attacks += 1

            trusted = filtered_updates


            if round == 24:    
                saved_vecs = [v.detach().clone().cpu().numpy() for v in vecs]
                saved_flags = attack_flags.copy()


            if round == 20:
              print(f"VAE threshold initialized at {threshold:.4f}")

            #


        if not use_vae:
          bypassed_attacks = total_attacks

        ##################################################
        # ROLLBACK
        ##################################################
        if use_rollback:

            # if len(trusted) < max(2, 0.1 * len(updates)):
            if len(trusted) < max(3, 0.2 * len(updates)):
                server_weights = prev_w
                print("Rollback triggered")
                continue

        if len(trusted) == 0:
            print(f"Warning: No trusted updates left for aggregation in round {round}. Skipping aggregation.")
            continue


        ##################################################
        # AGGREGATION
        ##################################################
        agg = {}
        for k in server_weights:

            if use_trimmed :
              layer = [u[k].flatten() for u in trusted]
              agg[k] = trimmed_mean(layer).view(server_weights[k].shape)
            else:
              layer = [u[k].flatten() for u in trusted]
              agg[k] = fedavg(layer).view(server_weights[k].shape)

        for k in server_weights:
            server_weights[k] += agg[k]

        global_model.load_state_dict(server_weights)


        ##################################################
        # METRICS
        ##################################################
        p,r_,auc,t = evaluate(global_model, X_test, y_test)

        asr = bypassed_attacks / total_attacks if total_attacks > 0 else 0

        detection_rate = detected_attacks / total_attacks if total_attacks > 0 else 0

        # print(f"Round {round} | P:{p:.3f} R:{r_:.3f} AUC:{auc:.3f} ASR:{asr:.3f}")
        print(f"Round {round} | P:{p:.3f} R:{r_:.3f} AUC:{auc:.3f} T:{t:.3f} ASR:{asr:.3f}")

        precision_hist.append(p)
        recall_hist.append(r_)
        auc_hist.append(auc)
        asr_hist.append(asr)
        detection_rate_hist.append(detection_rate)


    if len(avg_scores_hist) > 0:
        plt.figure()

        plt.plot(avg_scores_hist, label="Average Score")

        plt.plot(threshold_hist, linestyle='--', label='Threshold')

        plt.title("Average VAE Score vs Threshold Over Rounds")
        plt.xlabel("Round")
        plt.ylabel("Score")
        plt.legend()
        plt.grid()
        plt.show()


    if 'saved_vecs' in locals():
        plot_pca(
            saved_vecs,
            saved_flags,
            title="PCA: Attack vs Normal Updates (Round 24)"
        )


    if 'saved_normal_scores' in locals():

        plt.figure()

        plt.hist(saved_normal_scores, bins=30, alpha=0.5, label="Normal")
        plt.hist(saved_attack_scores, bins=30, alpha=0.5, label="Attack")

        if saved_threshold is not None:
            plt.axvline(saved_threshold, linestyle='--', label='Threshold')

        plt.legend()
        plt.title("Score Distribution (Round 24)")
        plt.xlabel("Anomaly Score")
        plt.ylabel("Frequency")
        plt.show()

    return precision_hist, recall_hist, auc_hist, asr_hist, detection_rate_hist
############################################################
# RUN ALL + STORE RESULTS
############################################################

results = {}

print("\n🔴 FedAvg baseline")
results["FedAvg"] = run_experiment(False, False, False, False)

print("\n🟢 Trimmed Mean")
results["Trimmed"] = run_experiment(True, False, False, False)

print("\n🔵 VAE + Trimmed")
results["VAE"] = run_experiment(True, True, False, False)

print("\n🟣 FULL:  VAE + Trimmed + Rollback")
results["Full"] = run_experiment(True, True, True, False)


############################################################
# PLOTS
############################################################

# Comparison plots
for i, metric in enumerate(["Precision", "Recall", "AUC", "ASR"]):
# for i, metric in enumerate(["Precision", "Recall", "AUC"]):
    plt.figure()
    for name, vals in results.items():
        plt.plot(vals[i], label=name)
    plt.title(metric + " Comparison")
    plt.legend()
    plt.grid()
    plt.show()

# Final bars
labels = list(results.keys())
precision_final = [results[k][0][-1] for k in labels]
asr_final = [results[k][3][-1] for k in labels]

plt.figure()
plt.bar(labels, precision_final)
plt.title("Final Precision")
plt.show()

plt.figure()
plt.bar(labels, asr_final)
plt.title("Final ASR")
plt.show()

#
plt.figure()
for name, vals in results.items():
    plt.plot(vals[4], label=name)

plt.title("Detection Rate Comparison")
plt.legend()
plt.grid()
plt.show()

#

# Tradeoff
plt.figure()
for i, name in enumerate(labels):
    plt.scatter(asr_final[i], precision_final[i], label=name)

plt.xlabel("ASR")
plt.ylabel("Precision")
plt.legend()
plt.grid()
plt.title("Precision vs Attack Success")
plt.show()